# Raw data exploration — No Fluff Jobs

Checking the raw payload before writing the pipeline, to decide what goes into the two
output tables: `jobs` (one row per advert) and `skills` (one row per advert and skill).

In [1]:
import gzip
import json
from pathlib import Path

import pandas as pd

RAW = Path("..") / "data" / "raw" / "source=nofluffjobs"
SNAPSHOT = sorted(folder.name for folder in RAW.iterdir())[-1]
SNAPSHOT

'date=2026-08-08'

In [2]:
records = []

for path in sorted((RAW / SNAPSHOT).glob("page_*.json.gz")):
    with gzip.open(path, "rt", encoding="utf-8") as file:
        page = json.load(file)
    records += page["postings"]

len(records)

21909

## What one record looks like

In [3]:
example = records[0]

for key, value in example.items():
    print(key, "=", str(value)[:80])

id = senior-power-platform-deployment-manager-1dea-Remote
name = 1dea
location = {'places': [{'city': 'Remote', 'url': 'senior-power-platform-deployment-manager-
posted = 1785168005852
renewed = 1786204805852
title = Senior Power Platform Deployment Manager
logo = {'original': 'companies/logos/original/1deagroup_logo_20240213_152915.jpeg', 'jo
category = devops
seniority = ['Senior']
url = senior-power-platform-deployment-manager-1dea-remote
regions = ['pl']
fullyRemote = False
salary = {'from': 31920.0, 'to': 35280.0, 'type': 'b2b', 'currency': 'PLN', 'disclosedAt'
flavors = ['it']
topInSearch = False
highlighted = False
help4Ua = False
reference = KZPNSWTZ
searchBoost = True
onlineInterviewAvailable = True
tiles = {'values': [{'value': 'devops', 'type': 'category'}, {'value': 'Power Platform',


## How many adverts are there?

The number above counts records, not adverts. Two fields could be the id.

In [4]:
ids = [record["id"] for record in records]
references = [record["reference"] for record in records]

print("records          ", len(records))
print("distinct id      ", len(set(ids)))
print("distinct reference", len(set(references)))

records           21909
distinct id       20548
distinct reference 3035


`reference` gives 7x fewer values. Looking at one of them:

In [5]:
busiest = pd.Series(references).value_counts().index[0]
same_advert = [record for record in records if record["reference"] == busiest]

print("records with this reference:", len(same_advert))

for record in same_advert[:5]:
    print(record["id"], "|", record["title"], "|", record["salary"]["from"])

records with this reference: 50
specjalistka-specjalista-ds-bankowosci-infakt-Remote | Specjalistka / Specjalista ds. Bankowości | 7000.0
specjalistka-specjalista-ds-bankowosci-infakt-Kraków | Specjalistka / Specjalista ds. Bankowości | 7000.0
specjalistka-specjalista-ds-bankowosci-infakt-Warszawa | Specjalistka / Specjalista ds. Bankowości | 7000.0
specjalistka-specjalista-ds-bankowosci-infakt-Gdańsk | Specjalistka / Specjalista ds. Bankowości | 7000.0
specjalistka-specjalista-ds-bankowosci-infakt-Wrocław | Specjalistka / Specjalista ds. Bankowości | 7000.0


Same job, once per location. So `reference` is the advert and `id` is advert x
location — everything has to be collapsed on `reference` before counting, otherwise
adverts open in many cities are counted many times.

In [6]:
adverts = {}

for record in records:
    if record["reference"] not in adverts:
        adverts[record["reference"]] = record

len(adverts)

3035

## The `jobs` table

In [7]:
def city_of(record):
    """First real city, skipping remote and province-only entries."""
    for place in record["location"]["places"]:
        if place.get("provinceOnly"):
            continue
        if place.get("city") and place["city"] != "Remote":
            return place["city"]
    return None


def is_remote(record):
    return any(place.get("city") == "Remote" for place in record["location"]["places"])


rows = []

for reference, record in adverts.items():
    salary = record["salary"]
    seniority = record["seniority"]
    rows.append(
        {
            "reference": reference,
            "title": record["title"],
            "company": record["name"],
            "category": record["category"],
            "seniority": seniority[0] if seniority else None,
            "city": city_of(record),
            "remote": is_remote(record),
            "salary_from": salary.get("from"),
            "salary_to": salary.get("to"),
            "contract": salary.get("type"),
            "currency": salary.get("currency"),
            "disclosed": salary.get("disclosedAt"),
            "posted": pd.to_datetime(record["posted"], unit="ms"),
        }
    )

jobs = pd.DataFrame(rows)
jobs.head()

,reference,title,company,category,seniority,city,remote,salary_from,salary_to,contract,currency,disclosed,posted
0,KZPNSWTZ,Senior Power Platform Deployment Manager,1dea,devops,Senior,Warszawa,True,31920.0,35280.0,b2b,PLN,VISIBLE,2026-07-27 16:00:05.852
1,APPZ486Y,Data Architect,EcoVadis,data,Expert,Warsaw,True,30000.0,35000.0,permanent,PLN,VISIBLE,2026-08-06 15:38:18.341
2,I6JIW3JN,Tester Automatyzujący,Link Group,testing,Senior,Warszawa,False,16800.0,18480.0,b2b,PLN,VISIBLE,2026-07-21 15:35:54.494
3,YTFWLPSD,Remote Account Manager,Spribe OÜ,sales,Mid,NaN,True,12903.0,17204.0,b2b,PLN,VISIBLE,2026-08-03 15:35:18.962
4,WXWG7O8I,Senior SRE,Link Group,devops,Senior,Warsaw,False,38640.0,43680.0,b2b,PLN,VISIBLE,2026-08-03 15:26:25.952


## Salary

In [8]:
print(jobs["currency"].value_counts())
print()
print(jobs["disclosed"].value_counts())
print()
print(jobs["contract"].value_counts())

currency
PLN    3035
Name: count, dtype: int64

disclosed
VISIBLE    3035
Name: count, dtype: int64

contract
b2b          2166
permanent     825
zlecenie       37
intern          4
uod             3
Name: count, dtype: int64


Everything is PLN per month — no currency conversion needed.

Every advert has a visible salary, which is an artefact of the collector: the request
sends `salaryCurrency=PLN&salaryPeriod=month` and the API uses those as filters. So the
data covers adverts that publish a salary, not the whole market. Needs a line in the
README, not a fix.

In [9]:
jobs["salary_from"].describe().round(0)

count     3035.0
mean     20323.0
std       7736.0
min          0.0
25%      15120.0
50%      20240.0
75%      25200.0
max      68880.0
Name: salary_from, dtype: float64

In [10]:
print("no upper bound:", jobs["salary_to"].isna().sum())
print("zero lower bound:", (jobs["salary_from"] == 0).sum())

no upper bound: 2
zero lower bound: 4


Both are tiny. Drop the zero-salary rows in the build script, use `salary_from`.

## Seniority, city, remote

In [11]:
jobs["seniority"].value_counts()

seniority
Senior     1610
Mid        1172
Junior      124
Expert      118
Trainee      11
Name: count, dtype: int64

Five clean values — a simple mapping, no text parsing.

In [12]:
jobs["city"].value_counts().head(15)

city
Warszawa               823
Kraków                 651
Wrocław                260
Warsaw                 198
Budapest               111
Katowice                70
Gdańsk                  69
Poznań                  66
Łódź                    24
Gdynia                  24
Cracow                  17
Gorzów Wielkopolski     17
Poznan                  13
niemcy                  11
Gliwice                 10
Name: count, dtype: int64

In [13]:
print(jobs["remote"].value_counts())
print()
print("adverts with no city:", jobs["city"].isna().sum())

remote
False    1883
True     1152
Name: count, dtype: int64

adverts with no city: 484


Warszawa and Warsaw are separate rows, same for Kraków / Cracow and Poznań / Poznan —
needs a small spelling map. Budapest is in the top ten, so filter on the country code
too. Adverts with no city are all remote.

## The `skills` table

In [14]:
skill_rows = []

for reference, record in adverts.items():
    for tile in record["tiles"]["values"]:
        if tile["type"] == "requirement":
            skill_rows.append({"reference": reference, "skill": tile["value"]})

skills = pd.DataFrame(skill_rows)
skills.head()

,reference,skill
0,KZPNSWTZ,Power Platform
1,KZPNSWTZ,CD
2,KZPNSWTZ,TFS
3,APPZ486Y,Architecture
4,APPZ486Y,Databricks


In [15]:
print("skill rows      ", len(skills))
print("distinct skills ", skills["skill"].nunique())
print("adverts covered ", skills["reference"].nunique(), "of", len(jobs))

skill rows       8279
distinct skills  1522
adverts covered  2933 of 3035


In [16]:
skills["skill"].value_counts().head(25)

skill
Python           422
Java             390
SQL              233
AI               168
Jira             118
.NET             117
Cloud            116
AWS              104
React            102
TypeScript       100
Azure             98
C#                87
K8s               87
Spring Boot       87
JavaScript        86
Linux             80
REST API          80
API               76
Doświadczenie     76
SAP               66
Confluence        62
Degree            61
MS Office         60
GCP               59
Angular           58
Name: count, dtype: int64

Plain strings, no proficiency level — that only exists on the individual job page.
The list mixes technologies with soft skills and umbrella terms, so a grouping file
will be needed later.

## Sanity check

In [17]:
jobs.groupby("seniority")["salary_from"].agg(["count", "median"]).round(0)

,count,median
seniority,,
Expert,118,26040.0
Junior,124,7287.0
Mid,1172,16800.0
Senior,1610,23000.0
Trainee,11,4300.0


In [18]:
top_cities = jobs["city"].value_counts().head(8).index
by_city = jobs[jobs["city"].isin(top_cities)]

by_city.groupby("city")["salary_from"].agg(["count", "median"]).round(0)

,count,median
city,,
Budapest,111,12915.0
Gdańsk,69,20160.0
Katowice,70,16800.0
Kraków,651,23000.0
Poznań,66,15000.0
Warsaw,198,21840.0
Warszawa,823,21840.0
Wrocław,260,18950.0


## Findings

1. ~3,000 adverts per snapshot, not 22,000 — collapse on `reference`.
2. All PLN per month, so no exchange rates needed.
3. Salary-disclosed adverts only — mention it in the README.
4. Skills have no level.
5. Cities need a spelling map and a Poland filter; seniority and contract type are clean.
6. Drop zero salaries.

Next: `src/build.py` doing the same over every snapshot, writing `jobs.csv` and
`skills.csv` with a `snapshot_date` column.